# 03 - Uplift models: T-Learner and X-Learner

Two meta-learners that estimate the conditional average treatment effect
(CATE), `tau(x) = E[Y(1) - Y(0) | X = x]`, rather than plain response
probability.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

from src.data import PRIMARY_OUTCOME, TREATMENT_COLUMN, load_parquet
from src.preprocessing import LightGBMFeatureTransform
from src.models import fit_t_learner, fit_x_learner
from src.evaluation import evaluate_ranking

PROCESSED_DIR = REPO_ROOT / "data" / "processed"
train_frame = load_parquet(PROCESSED_DIR / "train.parquet")
val_frame = load_parquet(PROCESSED_DIR / "validation.parquet")

transform = LightGBMFeatureTransform()
X_train = transform.fit_transform(train_frame)
X_val = transform.transform(val_frame)

Y_train, Y_val = train_frame[PRIMARY_OUTCOME], val_frame[PRIMARY_OUTCOME]
T_train, T_val = train_frame[TREATMENT_COLUMN], val_frame[TREATMENT_COLUMN]

## T-Learner

Two independent outcome models, one per arm: `mu1_hat(x) = E[Y | T=1, X=x]`,
`mu0_hat(x) = E[Y | T=0, X=x]`, and `tau_hat(x) = mu1_hat(x) - mu0_hat(x)`.
The simplest CATE estimator, and the required baseline this project compares
X-Learner and Causal Forest against.

In [ ]:
t_learner = fit_t_learner(X_train, T_train, Y_train, X_val, T_val, Y_val, seed=42)
t_learner.mu1.best_iteration, t_learner.mu0.best_iteration

In [ ]:
t_learner_scores = t_learner.predict_tau(X_val)
t_learner_ranking = evaluate_ranking(t_learner_scores, T_val, Y_val)
print("T-Learner qini_above_random:", round(t_learner_ranking.qini_above_random, 6))
t_learner_ranking.uplift_at_k

## X-Learner

Designed to exploit arm-size imbalance (CRITEO-UPLIFTv2.1 has far more
treated than control rows). Two-stage:

1. **Nuisance stage** -- `mu1_hat`, `mu0_hat`, fit with two-fold
   cross-fitting. Every row's out-of-fold (OOF) nuisance prediction comes
   from the model fit on the *opposite* fold.
2. **Effect stage** -- pseudo-outcomes `D1 = Y - mu0_oof` (on treated rows)
   and `D0 = mu1_oof - Y` (on control rows), each regressed on `X` to give
   `tau1_hat`, `tau0_hat`. Final estimate: a propensity-weighted combination
   of the two.

**Correctness note:** the categorical feature transform for the nuisance
stage must be fit fold-locally -- only on each fold's own training rows, not
on the whole train partition. `fit_x_learner` in `src/models.py` does this
internally; fitting one global transform and reusing it across both folds
would leak each fold's categorical vocabulary into the other (opposite) fold.
This was a real, fixed bug in this project's development.

In [ ]:
x_learner = fit_x_learner(X_train, T_train, Y_train, seed=42)
x_learner_scores = x_learner.predict_tau(X_val)
x_learner_ranking = evaluate_ranking(x_learner_scores, T_val, Y_val)
print("X-Learner qini_above_random:", round(x_learner_ranking.qini_above_random, 6))
x_learner_ranking.uplift_at_k

## Compare

In [ ]:
import matplotlib.pyplot as plt

for label, ranking in [("T-Learner", t_learner_ranking), ("X-Learner", x_learner_ranking)]:
    plt.plot(ranking.qini_curve["coverage"], ranking.qini_curve["qini_gain"], label=label)
plt.plot([0, 1], [0, t_learner_ranking.theoretical_random_qini_area * 2], "--", label="Theoretical random")
plt.xlabel("Population coverage")
plt.ylabel("Cumulative incremental conversions")
plt.legend()
plt.title("T-Learner vs X-Learner: Qini curves")

## Next

Continue with `04_causal_forest.ipynb`.